# Chương 4. HỌC SÂU

## Mô hình AlexNet

Kiến trúc AlexNet, do Alex Krizhevsky và cộng sự giới thiệu năm 2012 trong bài “ImageNet Classification with Deep Convolutional Neural Networks”, đã giành chiến thắng tại cuộc thi ILSVRC.

<p align="center">
  <img src="../picture/AlexNet Architecture.png" width="800">
  <br>
  <em>Hình 1. Kiến trúc AlexNet được đề xuất trong ImageNet Classification with Deep Convolutional Neural Networks.</em>
</p>


In [8]:
# Kiểm tra đã nhận Cuda
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

# Local Response Normalization (LRN)
def LRN(x):
    return tf.nn.local_response_normalization(
        x, depth_radius=2, bias=1.0, alpha=1e-4, beta=0.75
    )

def group_conv(x, filters, kernel):
    x1, x2 = tf.split(x, 2, axis=3)
    y1 = layers.Conv2D(filters, kernel, padding='same', activation='relu')(x1)
    y2 = layers.Conv2D(filters, kernel, padding='same', activation='relu')(x2)
    return layers.Concatenate(axis=3)([y1, y2])

def alexnet():
    inp = Input(shape=(227, 227, 3))

    # Conv1
    x = layers.Conv2D(96, (11, 11), strides=4, activation='relu')(inp)
    x = layers.Lambda(LRN)(x)
    x = layers.MaxPooling2D(3, strides=2)(x)

    # Conv2 (group=2)
    x = group_conv(x, 128, (5, 5))
    x = layers.Lambda(LRN)(x)
    x = layers.MaxPooling2D(3, strides=2)(x)

    # Conv3
    x = layers.Conv2D(384, (3, 3), padding='same', activation='relu')(x)

    # Conv4 (group=2)
    x = group_conv(x, 192, (3, 3))

    # Conv5 (group=2)
    x = group_conv(x, 128, (3, 3))
    x = layers.MaxPooling2D(3, strides=2)(x)

    # Fully Connected 
    x = layers.Flatten()(x)
    x = layers.Dense(4096, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(4096, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1000, activation='softmax')(x)

    return models.Model(inputs=inp, outputs=out)

model = alexnet()
model.summary()


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 227, 227, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_13 (Conv2D)             (None, 55, 55, 96)   34944       ['input_2[0][0]']                
                                                                                                  
 lambda_5 (Lambda)              (None, 55, 55, 96)   0           ['conv2d_13[0][0]']              
                                                                                                  
 max_pooling2d_6 (MaxPooling2D)  (None, 27, 27, 96)  0           ['lambda_5[0][0]']         

## Ứng dụng trên dữ liệu thực tế

### Downloand dữ liệu

In [16]:
import os
import kagglehub
import shutil

# Đường dẫn bạn muốn lưu dataset
save_path = r"..\picture"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(save_path, exist_ok=True)

# Tải dataset bằng kagglehub
download_path = kagglehub.dataset_download("tongpython/cat-and-dog")
print("Downloaded to:", download_path)

# Copy toàn bộ file/folder sang đường dẫn mới
for item in os.listdir(download_path):
    src = os.path.join(download_path, item)
    dst = os.path.join(save_path, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("\n✅ Dataset copied successfully to:")
print(save_path)

# Liệt kê file trong thư mục đích
print("\n📂 Files inside the dataset folder:")
print(os.listdir(save_path))


Downloaded to: C:\Users\Admin\.cache\kagglehub\datasets\tongpython\cat-and-dog\versions\1

✅ Dataset copied successfully to:
..\picture

📂 Files inside the dataset folder:
['AlexNet Architecture.png', 'cute_black_cat.jpg', 'test_set', 'training_set']
